# Project Journal 02: Real Sleipner Intake and Constraint Setup

This entry continues the running `CO2Shift` journal. It documents the point where the project stopped relying on pseudo-field checks and started using real public Sleipner exports plus benchmark-derived structural constraints.


## Context

Journal 01 ended with a workable synthetic benchmark and a manifest-ready field path, but no real field evidence.

This stage moved the project into a more serious regime:

- real `94p07`, `01p07`, `04p07`, and `06p07` exported inline sections were brought into the workflow
- the public Sleipner benchmark model was used to derive a storage-interval mask on the exact inline geometry
- field postprocessing was added so the model output could be judged against geological plausibility instead of only synthetic metrics

The goal of this stage was not to prove the model had solved field monitoring. The goal was to establish a defensible field-evaluation protocol.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

ROOT = Path.cwd().resolve()
if not (ROOT / 'runs').exists():
    ROOT = ROOT.parent.resolve()

def load_json(path):
    with Path(path).open('r', encoding='utf-8') as handle:
        return json.load(handle)

def show_images(image_paths, titles, figsize=(18, 8)):
    fig, axes = plt.subplots(1, len(image_paths), figsize=figsize)
    if len(image_paths) == 1:
        axes = [axes]
    for ax, image_path, title in zip(axes, image_paths, titles):
        image = mpimg.imread(image_path)
        ax.imshow(image)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

ROOT

p07_summary = load_json(ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'summary.json')
p07_metrics = load_json(ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'metrics.json')
field = p07_summary['Field']
field


## What Changed This Stage

The main additions in this stage were:

- real Sleipner field sections were exported into `.npy` arrays and loaded through the manifest path
- a benchmark-derived storage-interval mask was aligned to the same inline geometry
- field predictions gained a constrained postprocessing path with shared thresholding, uncertainty gating, morphology cleanup, and reservoir masking
- the field review expanded from a single pair to a multi-vintage `p07` sequence so temporal plausibility could be checked

This stage made the project much more honest. Instead of asking whether the model looked good on a pseudo-field example, it asked whether the field output stayed inside the storage interval and evolved in a believable way across vintages.


In [ ]:
def pair_summary_table(field_summary):
    rows = []
    for pair_name, metrics in field_summary['pairs'].items():
        rows.append({
            'pair': pair_name,
            'cross_eq_trace_iou_2010': metrics['cross_equalized_difference']['trace_iou_with_2010_support'],
            'cross_eq_trace_fraction': metrics['cross_equalized_difference']['predicted_trace_fraction'],
            'raw_hybrid_outside_reservoir': metrics['hybrid_ml']['outside_reservoir_fraction'],
            'constrained_trace_iou_2010': metrics['hybrid_ml_constrained']['trace_iou_with_2010_support'],
            'constrained_trace_fraction': metrics['hybrid_ml_constrained']['predicted_trace_fraction'],
            'constrained_predicted_fraction': metrics['hybrid_ml_constrained']['predicted_fraction'],
            'constrained_outside_reservoir': metrics['hybrid_ml_constrained']['outside_reservoir_fraction'],
        })
    return pd.DataFrame(rows)

pair_summary_table(field)


## Current Results

The field story became more believable in this stage, but only in a narrow and honest sense.

What the evidence supports:

- the constrained hybrid output is much more selective than the cross-equalized difference baseline
- the constrained trace-support overlap with the public 2010 plume envelope grows across the `2001 -> 2004 -> 2006` sequence
- constrained outputs stay inside the benchmark storage interval

What the evidence does not support:

- the raw hybrid model is not field-ready on its own
- zero outside-reservoir leakage is achieved because the benchmark mask is enforced in postprocessing
- constrained pixel area is still not the cleanest field metric for this sequence

That is why the field claim at this stage should be framed around lateral plume-support behavior, not raw plume segmentation quality.


In [ ]:
show_images(
    [
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2001_inline_1840_mid_cross_equalized_difference.png',
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2004_inline_1840_mid_cross_equalized_difference.png',
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2006_inline_1840_mid_cross_equalized_difference.png',
    ],
    ['2001 cross-equalized baseline', '2004 cross-equalized baseline', '2006 cross-equalized baseline'],
    figsize=(18, 6),
)

show_images(
    [
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2001_inline_1840_mid_hybrid_constrained.png',
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2004_inline_1840_mid_hybrid_constrained.png',
        ROOT / 'runs' / 'sleipner_manifest' / 'results' / 'figures' / 'field_sleipner_2006_inline_1840_mid_hybrid_constrained.png',
    ],
    ['2001 constrained hybrid', '2004 constrained hybrid', '2006 constrained hybrid'],
    figsize=(18, 6),
)


## Open Problems

Several important limitations remained after this stage:

- the raw hybrid model still leaked strongly outside the reservoir before constraining
- the 2010 plume-support envelope is a later-time structural reference for the earlier `p07` vintages, not exact ground truth
- the constrained pixel area did not give a fully stable monotone progression
- the project still needed a stronger field-facing audit than just comparing constrained maps against a later support envelope

Those limits are why this stage should be treated as a field-intake and plausibility milestone, not a final field-validation claim.


In [ ]:
temporal = field['temporal_consistency']
progress = pd.DataFrame({
    'pair': temporal['ordered_pairs'],
    'constrained_trace_fraction': [temporal['constrained_trace_fraction_by_pair'][name] for name in temporal['ordered_pairs']],
    'constrained_support_iou_2010': [temporal['constrained_support_iou_by_pair'][name] for name in temporal['ordered_pairs']],
})

print(field['support_note'])
display(progress)
print('Area non-decreasing:', temporal['constrained_area_non_decreasing'])
print('Trace fraction non-decreasing:', temporal['constrained_trace_fraction_non_decreasing'])
print('Support IoU non-decreasing:', temporal['constrained_support_iou_non_decreasing'])


## Next Stage

The next defensible step after this stage was to stress-test the field story rather than keep tuning the model.

Immediate follow-up questions were:

1. Does the same logic still hold against a stronger classical field baseline?
2. Does the constrained hybrid align well with a direct same-year 2010 benchmark check, not just an earlier-vintage support envelope comparison?
3. Can the repo save that field audit with explicit provenance instead of mixing new outputs with old training artifacts?

Series note:

- This notebook captures the real field-intake phase.
- The next entry should focus on the stronger field audit and benchmark-alignment evidence.
